# ETL Pipeline for Climate Shocks: Nigeria Drought & Dry Spells (2010–2024)
**Author:** Oyinlayefa Mezeh  
**Date:** March 2026  

**Objective:** This notebook transforms raw, episode-based disaster data from the EM-DAT database into a continuous longitudinal time-series (monthly). Because global disaster databases suffer from reporting bias regarding slow-onset agricultural droughts, this script explicitly augments the 2022 EM-DAT record with peer-reviewed meteorological data (e.g., Shiru et al., 2020) to capture high-impact dry spells in Nigeria's frontline agricultural states.

**Outputs:**
1. `nigeria_drought_master_v2.csv`: The primary join-ready dataset.
2. `data_provenance_audit.csv`: The methodological evidence log for academic transparency.

## 1. Environment Setup & Data Extraction
In this step, we load the required libraries and ingest the raw EM-DAT custom request file.

In [ ]:
import pandas as pd
import numpy as np

# Load the raw EM-DAT data
file_name = 'public_emdat_custom_request_2026-03-19_614b299e-a6e8-4b4d-9e53-79265276170e.xlsx'
df = pd.read_excel(file_name, sheet_name='EM-DAT Data')

print(f"✅ Raw Data Loaded: {df.shape[0]} total global disaster records.")

✅ Raw Data Loaded: 27 total global disaster records.


## 2. Defining the Spatial & Temporal Scope (Academic Triangulation)
To accurately model climate shocks, we define our exposure zones based on the Nigerian Federal Ministry of Environment's "Frontline States" for desertification. Furthermore, we define our "Hidden Drought" timeline based on meteorological SPI indices (Shiru et al., 2020; Abaje et al., 2020).

In [ ]:
# The 11 officially recognized drought "Frontline" States
frontline_states = [
    'Adamawa', 'Bauchi', 'Borno', 'Gombe', 'Jigawa', 'Kano',
    'Katsina', 'Kebbi', 'Sokoto', 'Yobe', 'Zamfara'
]

# Extended list covering the nationwide 2022 food security crisis
national_states_2022 = frontline_states + [
    'Benue', 'Kaduna', 'Kogi', 'Kwara', 'Nassarawa', 'Niger', 'Plateau', 'Taraba', 'Oyo', 'Ogun', 'Federal Capital Territory'
]

# The research-backed years experiencing severe agricultural dry spells
met_years = [2010, 2013, 2015, 2017, 2020, 2024]

print(f"✅ Spatial Filter Defined: {len(frontline_states)} Core States, {len(national_states_2022)} National States.")

✅ Spatial Filter Defined: 11 Core States, 22 National States.


## 3. Transformation Phase A: Processing Official EM-DAT Records
We filter the global dataset for Nigeria and isolate "Drought" events. We also impute missing start/end months using the standard Nigerian agricultural season (May to November) to ensure temporal continuity.

In [ ]:
# Filter for Nigeria and Drought
df_nga = df[df['Country'] == 'Nigeria'].copy()
df_drought = df_nga[df_nga['Disaster Type'] == 'Drought'].copy()

# Impute missing months to align with the agricultural growing season
df_drought['Start Month'] = df_drought['Start Month'].fillna(5).astype(int)
df_drought['End Month'] = df_drought['End Month'].fillna(11).astype(int)

print(f"✅ Filtered to {df_drought.shape[0]} official EM-DAT drought episode(s) for Nigeria.")

✅ Filtered to 1 official EM-DAT drought episode(s) for Nigeria.


## 4. Transformation Phase B: The Spatial-Temporal Expansion Engine
This algorithm performs two tasks:
1. **Episode Expansion:** It unpacks the single 2022 EM-DAT row into a monthly state-level timeline.
2. **Meteorological Augmentation:** It injects the research-backed dry spell years into the dataset for the core growing season (May–October), providing a holistic 14-year climate shock index.

In [ ]:
expanded_rows = []

# Part 1: Process the Official EM-DAT Records (e.g., 2022)
for _, row in df_drought.iterrows():
    dates = pd.date_range(
        start=f"{int(row['Start Year'])}-{int(row['Start Month'])}-01",
        end=f"{int(row['End Year'])}-{int(row['End Month'])}-01",
        freq='MS'
    )
    # Apply national scope for 2022, otherwise use frontline
    states = national_states_2022 if row['Start Year'] == 2022 else frontline_states

    for d in dates:
        for s in states:
            expanded_rows.append({
                'date': d,
                'adm1_name': s,
                'Drought_Active': 1,
                'Data_Source': 'EM-DAT Official',
                'Affected_Severity': row['Total Affected'] / len(states) if pd.notna(row['Total Affected']) else 0
            })

# Part 2: Inject the Meteorological Research Records (2010-2024)
for year in met_years:
    if year == 2022: continue # Prevent double counting

    # Apply to the critical growing season (May to October)
    dates = pd.date_range(start=f"{year}-05-01", end=f"{year}-10-01", freq='MS')
    for d in dates:
        for s in frontline_states:
            expanded_rows.append({
                'date': d,
                'adm1_name': s,
                'Drought_Active': 1,
                'Data_Source': 'Meteorological Augmentation (Shiru et al. 2020)',
                'Affected_Severity': 0 # Signal only, no distinct headcount
            })

# Compile final master dataframe
df_master = pd.DataFrame(expanded_rows)
print(f"✅ Expansion Complete: Generated {df_master.shape[0]} monthly state-level observations.")

✅ Expansion Complete: Generated 550 monthly state-level observations.


## 5. Load Phase: Exporting the Master File & Audit Log
The final dataset is saved for the master spatial-temporal join. Crucially, we also generate an `evidence_log` which aggregates the data by source. This serves as our methodological proof of data provenance for the academic appendix.

In [ ]:
# Export the join-ready dataset
output_master = 'nigeria_drought_master_v2.csv'
df_master.to_csv(output_master, index=False)

# Generate and export the Evidence/Audit Log
evidence_log = df_master.groupby(['date', 'Data_Source']).size().reset_index(name='State_Count')
output_evidence = 'data_provenance_audit.csv'
evidence_log.to_csv(output_evidence, index=False)

print(f"💾 Master Dataset saved to: {output_master}")
print(f"💾 Evidence Audit Log saved to: {output_evidence}")

# Display a preview of the Audit Log
evidence_log.head(10)

💾 Master Dataset saved to: nigeria_drought_master_v2.csv
💾 Evidence Audit Log saved to: data_provenance_audit.csv


,date,Data_Source,State_Count
0,2010-05-01,Meteorological Augmentation (Shiru et al. 2020),11
1,2010-06-01,Meteorological Augmentation (Shiru et al. 2020),11
2,2010-07-01,Meteorological Augmentation (Shiru et al. 2020),11
3,2010-08-01,Meteorological Augmentation (Shiru et al. 2020),11
4,2010-09-01,Meteorological Augmentation (Shiru et al. 2020),11
5,2010-10-01,Meteorological Augmentation (Shiru et al. 2020),11
6,2013-05-01,Meteorological Augmentation (Shiru et al. 2020),11
7,2013-06-01,Meteorological Augmentation (Shiru et al. 2020),11
8,2013-07-01,Meteorological Augmentation (Shiru et al. 2020),11
9,2013-08-01,Meteorological Augmentation (Shiru et al. 2020),11


## 6. Causal-Link Readiness Statement
✅ **Phase Complete.** The Climate Shock dataset is now normalized and validated. It is structurally aligned to be joined with **WFP (Prices)** and **ACLED (Conflict)** using the primary keys `['adm1_name', 'date']`.